In [21]:
import arcpy
import os
from arcpy.sa import ExtractMultiValuesToPoints
import math

# check out Spatial Analyst
arcpy.CheckOutExtension("Spatial")

'CheckedOut'

In [22]:
# ── PATHS ──────────────────────────────────────────────────────────────────────────────────────────────────

# Input bathymetry and slope rasters
bathy_p26920_v6642 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\BATHY_USC10mGBC15a_p26920_v6642"
slope_p26920_v6642 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\SLOPE_USC10mGBC15a_p26920_v6642"

# Input LF/HF point feature classes for GIS calculations of mission background conditions
lf_points_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_LF_background_points_p26920"
hf_points_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_HF_background_points_p26920"

# Input USC bathy raster footprint for populating "native_raster_resolution_m" column in point feature class Attribute Tables 
# ***Only use footprint from bathy raster - USC bathy/slope rasters have exact same resolution + bathy/slope footprints have exact same xy extent, domain, resolution, and tolerance; only creating one raster res column per Attribute Table and output res value for each row will be the same using either bathy or slope USC footprints
usc_fp_p26920_v6642 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\USC_footprint_mosaic_bathy_p26920_v6642"

In [23]:
# ── SETTINGS ───────────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True

In [24]:
############### NOTE ###############
# 'Extract Multi Values to Points' tool edits Attribute Table of input feature class in-place (not additional output files/etc.)
# Edits are saved to source feature class in .gdb (not just to layer on map)
####################################

In [ ]:
# ── 1. CALCULATE BATHYMETRIC DEPTH AND SLOPE FOR EACH MISSION BACKGROUND POINT ─────────────────────────────

# Extract Multi Values to Points (Spatial Analyst Tools): Function
def extract_bathy_slope(in_points, bathy_raster, slope_raster):
    try:
        ExtractMultiValuesToPoints(
            in_point_features=in_points,
            in_rasters=[
                # output values calculated from raster 'bathy_p26920_v6642' to field new field "bathymetric_depth_m"
                [bathy_raster, "bathymetric_depth_m"],
                # output values calculated from raster 'slope_p26920_v6642' to field new field "bathymetric_slope_deg"
                [slope_raster, "bathymetric_slope_deg"],
            ],
            # values calculated from adjacent cells with valid values (NoData values ignored); bathy and slope are continuous variables so calculations across gradient (bilinear) are better representation of data than single value from center of raster grid cell (nearest neighbor)
            bilinear_interpolate_values="BILINEAR"
        )
        # print confirmation that tool ran successfully (EVEN IF some cells in table did NOT recieve a bathy/slope value)
        print(f"✅ Extracted bathymetry and slope values for all rows in {os.path.basename(in_points)}")
    # print warning message if tool fails entirely
    except Exception as e:
        print(f"\n❌ [FAILURE] Extract Multi Values to Points failed for {os.path.basename(in_points)}: {e} \n")

# call function for LF and HF mission points feature classes
extract_bathy_slope(lf_points_p26920, bathy_p26920_v6642, slope_p26920_v6642)
extract_bathy_slope(hf_points_p26920, bathy_p26920_v6642, slope_p26920_v6642)

✅ Extracted bathymetry and slope values for all rows in calc_LF_background_points_p26920
✅ Extracted bathymetry and slope values for all rows in calc_HF_background_points_p26920


In [26]:
# ── CHECK FOR NULL/EMPTY CELLS IN OUTPUT BATHY/SLOPE FIELDS ────────────────────────────────────────────────

# list of fields created by ExtractMultiValuesToPoints
check_fields = ["bathymetric_depth_m", "bathymetric_slope_deg"]

# Loop through and check data in Attribute Tables of each point feature class (LF and HF)
for fc in [lf_points_p26920, hf_points_p26920]:
    # count and report total n records (rows) in feature class table
    total = int(arcpy.management.GetCount(fc)[0])
    print(f"\n------ Report for {os.path.basename(fc)} ------")
    print(f"  Total rows in Attribute Table: {total}")
    # loop through each of the two fields created by ExtractMultiValuesToPoints (in 'check_fields' list defined above)
    for field in check_fields:
        # count rows where extracted value is Null (*this may happen along border between USC and GBC rasters but should be rare)
        null_count = 0
        # collect object IDs of any null rows for inspection
        null_oids = []
        with arcpy.da.SearchCursor(fc, ["OID@", field]) as cursor:
            for oid, value in cursor:
                # NoData/unmatched points returned as 'None' or 'NaN'
                if value is None or (isinstance(value, float) and math.isnan(value)):
                    null_count += 1
                    null_oids.append(oid)
        # report presence/absence of Null or empty cells in fields created by ExtractMultiValuesToPoints (in 'check_fields' list defined above)
        if null_count == 0:
            print(f"  {field}: ✅ NO null values")
        else:
            print(f"  {field}: ⚠️[WARNING] {null_count} NULL value(s) at OID(s) {null_oids}")


 ------ Report for calc_LF_background_points_p26920 ------
  Total rows in Attribute Table: 22497
  bathymetric_depth_m: ✅ NO null values
  bathymetric_slope_deg: ⚠️[WARNING] 2 NULL value(s) at OID(s) [3058, 4843]

 ------ Report for calc_HF_background_points_p26920 ------
  Total rows in Attribute Table: 4319
  bathymetric_depth_m: ✅ NO null values
  bathymetric_slope_deg: ✅ NO null values


In [27]:
# ── 2. ADD + POPULATE COLUMN TO TRACK NATIVE RESOLUTION OF RASTER USED FOR BATHY/SLOPE CALCULATIONS ─────────

# Function to add "native_raster_resolution_m" field to point feature class Attribute Tables, select points inside/outside USC footprint polygon, and populate "native_raster_resolution_m" field based on overlap with USC footprint
def assign_raster_resolution(in_points, footprint_polygon):
    field_name = "native_raster_resolution_m"

    # add field only if it does NOT already exist (**if field IS already present, any data it contains will be overwritten by Calculate Field tool)
    existing_fields = [f.name for f in arcpy.ListFields(in_points)]
    if field_name not in existing_fields:
        arcpy.management.AddField(
            in_table=in_points,
            field_name=field_name,
            # set field data type to "SHORT" (16-bit signed integer that can store integers within range -32,768 to 32,767)
            field_type="SHORT"
        )

    # create temporary feature layer object from point feature class (in-memory layer object that *can hold selections* created by SelectLayerByLocation/SelectLayerByAttribute tools (source feature class files cannot hold selections) - selections in feature layer determine which rows get written in Attribute Table of original feature class file)
    layer = "points_lyr"
    arcpy.management.MakeFeatureLayer(
        in_features=in_points,
        out_layer=layer
    )

    ########################### points WITHIN footprint of USC raster (10-m resolution) ###########################

    # Select Layer by Location (Data Management Tools): Select all points WITHIN (completely or partially) USC footprint
    arcpy.management.SelectLayerByLocation(
        # use feature layer created from point feature class
        in_layer=layer,
        # *"INTERSECT" default includes points completely AND partially in USC footprint
        overlap_type="INTERSECT",
        select_features=footprint_polygon,
        # "NEW_SELECTION" clears any existing selections in feature layer and starts fresh with only features matching current query parameters
        selection_type="NEW_SELECTION"
    )
    # populate "native_raster_resolution_m" field of selected rows (within USC fp) with '10' (CalculateField writes data to original feature class file for ONLY rows held in feature layer selection)
    inside_count = int(arcpy.management.GetCount(layer)[0])
    arcpy.management.CalculateField(
        in_table=layer,
        field=field_name,
        expression=10,
        expression_type="PYTHON3"
    )

    ########################### points OUTSIDE footprint of USC raster (~473-m resolution) ###########################

    # Select Layer by Attribute (Data Management Tools): Select all points OUTSIDE of USC footprint
    arcpy.management.SelectLayerByAttribute(
        in_layer_or_view=layer,
        # "SWITCH_SELECTION" DE-SELECTS all points in pre-existing selection + SELECTS all points in feature layer that were excluded from pre-existing selection (*exact inverse of selection created by 'Select Layer By Location' tool)
        selection_type="SWITCH_SELECTION"
    )
    # populate "native_raster_resolution_m" field of selected rows (outside USC fp) with '473' (CalculateField writes data to original feature class file for ONLY rows held in feature layer selection)
    outside_count = int(arcpy.management.GetCount(layer)[0])
    arcpy.management.CalculateField(
        in_table=layer,
        field=field_name,
        expression=473,
        expression_type="PYTHON3"
    )

    ########################### check data populated in "native_raster_resolution_m" field ###########################

    # clear selection so all rows are available for data checks below
    arcpy.management.SelectLayerByAttribute(
        in_layer_or_view=layer,
        selection_type="CLEAR_SELECTION")

    # print report header with input point feature class name + counts of rows with each raster resolution
    print(f"\n ------ Report for {os.path.basename(in_points)} ------")
    print(f"  INSIDE usc footprint (10 m): {inside_count}")
    print(f"  OUTSIDE usc footprint (473 m): {outside_count}")

    # check for any rows in "native_raster_resolution_m" that were NOT populated (still Null)
    null_count = 0
    null_oids = []
    with arcpy.da.SearchCursor(in_points, ["OID@", field_name]) as cursor:
        for oid, value in cursor:
            if value is None:
                null_count += 1
                null_oids.append(oid)
    if null_count == 0:
        print(f"  '{field_name}' field: ✅ ALL rows populated")
    else:
        print(f"  '{field_name}' field: ⚠️[WARNING] {null_count} unpopulated (NULL) value(s) at OID(s) {null_oids}")

    # report all unique values present in "native_raster_resolution_m" *should ouput [10, 473]*
    unique_vals = sorted({row[0] for row in arcpy.da.SearchCursor(in_points, [field_name])}, key=lambda x: (x is None, x))
    print(f"  Unique values in '{field_name}' field: {unique_vals} \n")

    # clean up feature layer
    arcpy.management.Delete(layer)


# call function for LF/HF mission points feature classes (in_points, footprint_polygon)
assign_raster_resolution(lf_points_p26920, usc_fp_p26920_v6642)
assign_raster_resolution(hf_points_p26920, usc_fp_p26920_v6642)


 ------ Report for calc_LF_background_points_p26920 ------
  INSIDE usc footprint (10 m): 19069
  OUTSIDE usc footprint (473 m): 3428
  native_raster_resolution_m: ✅ ALL rows populated
  Unique values in native_raster_resolution_m: [10, 473]

 ------ Report for calc_HF_background_points_p26920 ------
  INSIDE usc footprint (10 m): 3665
  OUTSIDE usc footprint (473 m): 654
  native_raster_resolution_m: ✅ ALL rows populated
  Unique values in native_raster_resolution_m: [10, 473]


In [28]:
# release Spatial Analyst license
arcpy.CheckInExtension("Spatial")

'CheckedOut'